# Intelligent Email Classifier with Llama (Google Colab version)

Every day, professionals wade through hundreds of emails, from urgent client requests to promotional offers. It's like trying to find important messages in a digital ocean. But AI can help you stay afloat by automatically sorting emails to highlight what matters most.

This notebook builds an intelligent email assistant using a local Llama model (via `llama-cpp-python`) to classify incoming emails into `Priority`, `Updates`, or `Promotions`.

### The Data
`email_categories_data.csv` has three columns:

| Column | Description |
|--------|-------------|
| email_id | A unique identifier for each email in the dataset. |
| email_content | The full email text including subject line and body. |
| expected_category | The correct classification: `Priority`, `Updates`, or `Promotions`. |

**Before running:** upload `email_categories_data.csv` using the file upload cell below, or mount your Google Drive and point `CSV_PATH` to it.

## 1. Install dependencies
Colab needs `llama-cpp-python` (for running the GGUF model) and `huggingface_hub` (to download the model weights, since Colab has no local model file like DataLab did).

In [ ]:
!pip install llama-cpp-python==0.2.82 huggingface_hub -q -q -q

## 2. Imports

In [ ]:
import pandas as pd
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from google.colab import files

## 3. Upload the dataset
Run this cell and select `email_categories_data.csv` from your computer. If you'd rather use Google Drive instead, skip this cell, run `from google.colab import drive; drive.mount('/content/drive')`, and set `CSV_PATH` to the file's Drive path.

In [ ]:
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]

In [ ]:
# Load the email dataset
emails_df = pd.read_csv(CSV_PATH)
print("Preview of our email dataset:")
emails_df.head(2)

## 4. Download the Llama model
Unlike DataLab, Colab doesn't have the `.gguf` file sitting locally, so we pull TinyLlama's quantized weights straight from the Hugging Face Hub the first time this runs (it's cached afterward for the rest of the session).

In [ ]:
model_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v0.3-GGUF",
    filename="tinyllama-1.1b-chat-v0.3.Q4_K_M.gguf"
)
print(f"Model downloaded to: {model_path}")

## 5. Load the model and define the prompt

In [ ]:
llm = Llama(model_path=model_path, verbose=False)

# Few-shot system prompt with one example per category
prompt = """You are an intelligent email classifier. Classify emails into Priority, Updates, or Promotions.

Example 1:
Urgent: Password Reset Required
Your account security requires immediate attention. Please reset your password within 24 hours.
Response: Priority

Example 2:
Special Offer - 50% Off Everything!
Don't miss our biggest sale of the year. Everything must go!
Response: Promotions

Example 3:
Canceled Event - Team Meeting
This event has been canceled and removed from your calendar.
Response: Updates

Example 4:
"""

## 6. Classification function

In [ ]:
def classify_message(llm, message, prompt):
    input_prompt = f"{prompt}\n{message}\nResponse:"
    response = llm(
        input_prompt,
        max_tokens=5,
        temperature=0,
        stop=["Q:", "\n"]
    )
    return response['choices'][0]['text'].strip()

## 7. Run classification on the dataset
This runs over the **full** dataset (not just the first two rows) and uses the model's **actual output** for each email — the original notebook had a bug where the final results were hardcoded to `"Priority"` and `"Promotions"` instead of using the real predictions. That's fixed here.

In [ ]:
results = []
for idx, row in emails_df.iterrows():
    email_content = row['email_content']
    expected_category = row['expected_category']

    model_output = classify_message(llm, email_content, prompt)

    results.append({
        'email_id': row.get('email_id', idx),
        'email_content': email_content,
        'expected_category': expected_category,
        'model_output': model_output
    })

results_df = pd.DataFrame(results)
results_df.head(10)

## 8. Evaluate accuracy
Compare the model's predictions against the known `expected_category` labels.

In [ ]:
results_df['correct'] = (
    results_df['model_output'].str.strip().str.lower()
    == results_df['expected_category'].str.strip().str.lower()
)

accuracy = results_df['correct'].mean()
print(f"Accuracy: {accuracy:.2%} ({results_df['correct'].sum()} / {len(results_df)} correct)")

print("\nMisclassified emails:")
results_df[~results_df['correct']][['email_id', 'expected_category', 'model_output']]

## 9. (Optional) Save results
Download the classification results as a CSV.

In [ ]:
results_df.to_csv('email_classification_results.csv', index=False)
files.download('email_classification_results.csv')